## Лабораторная работа №2
### Обработка пропусков, кодирование категориальных признаков, масштабирование данных

**Цель:** изучить базовые методы предварительной подготовки данных для последующего построения моделей.

**Задачи:**
1. Загрузить датасет с пропусками и категориальными переменными.
2. Провести анализ и обработку пропусков.
3. Выполнить кодирование категориальных признаков (Label Encoding, One-Hot Encoding).
4. Масштабировать числовые признаки (StandardScaler, MinMax).
5. Визуализировать изменения.

**Используемый датасет:** Titanic (см. описание ниже).

## 1. Описание датасета Titanic

Файл `titanic.csv` содержит информацию о пассажирах злополучного рейса.

**Признаки:**
- `PassengerId` – идентификатор (не несёт полезной информации, будет удалён).
- `Survived` – целевая переменная (1 – выжил, 0 – погиб).
- `Pclass` – класс билета (1, 2, 3) – порядковый категориальный признак.
- `Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin`, `Embarked` – прочие характеристики.

**Пропуски присутствуют в столбцах:** `Age` (много), `Cabin` (очень много), `Embarked` (2 пропуска).
**Категориальные признаки:** `Sex`, `Embarked`, `Pclass` (можно рассматривать как категориальный).

In [1]:
// Основные пакеты для анализа данных
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Microsoft.ML, 3.0.1"

// Пакеты для визуализации
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1"

// Для корреляции (может пригодиться)
#r "nuget: MathNet.Numerics, 5.0.0"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.Data.Analysis, 0.22.1 Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1 Microsoft.ML, 3.0.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\0\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\0\.nuget\packages\microsoft.data.analysis\0.22.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

In [2]:
using Plotly.NET;
using PlotlyCS = Plotly.NET.CSharp.Chart;
using Microsoft.Data.Analysis;
using Microsoft.ML;
using MathNet.Numerics.Statistics;
using Microsoft.DotNet.Interactive.Formatting;
using System.Linq;
using System.IO;
using System.Net;
using System.Globalization;

In [16]:
var dataDir = Path.Combine(Directory.GetCurrentDirectory(), "Data");
Directory.CreateDirectory(dataDir);
var dataPath = Path.Combine(dataDir, "titanic.csv");

if (!File.Exists(dataPath))
{
    var url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv";
    Console.WriteLine("Скачиваю titanic.csv...");
    new WebClient().DownloadFile(url, dataPath);
    Console.WriteLine("Готово.");
}

// Явно указываем имена и типы столбцов
string[] colNames = new[] { "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked" };
Type[] colTypes = new Type[] { typeof(float), typeof(float), typeof(float), typeof(string), typeof(string), typeof(float), typeof(float), typeof(float), typeof(string), typeof(float), typeof(string), typeof(string) };

var df = DataFrame.LoadCsv(dataPath,
    separator: ',',
    header: true,
    columnNames: colNames,
    dataTypes: colTypes,
    cultureInfo: CultureInfo.InvariantCulture);

df.Head(5).Display();

index,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,,S



(9,5): warning SYSLIB0014: "WebClient.WebClient()" является устаревшим: 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



In [17]:
Console.WriteLine("Общее количество строк: " + df.Rows.Count);
Console.WriteLine("Пропущенных значений по столбцам:");

foreach (var colName in df.Columns.Select(c => c.Name))
{
    var col = df[colName];
    long nullCount = 0;
    for (long i = 0; i < df.Rows.Count; i++)
    {
        if (col[i] is null || (col[i] is float f && float.IsNaN(f)))
            nullCount++;
    }
    Console.WriteLine($"{colName}: {nullCount} пропусков");
}

Общее количество строк: 891
Пропущенных значений по столбцам:
PassengerId: 0 пропусков
Survived: 0 пропусков
Pclass: 0 пропусков
Name: 0 пропусков
Sex: 0 пропусков
Age: 177 пропусков
SibSp: 0 пропусков
Parch: 0 пропусков
Ticket: 0 пропусков
Fare: 0 пропусков
Cabin: 0 пропусков
Embarked: 0 пропусков


## 2. Обработка пропусков

Выберем разные стратегии в зависимости от столбца:
- **Age** – заполним медианой (числовой признак).
- **Embarked** – заполним модой (категориальный признак).
- **Cabin** – удалим целиком (слишком много пропусков, почти 80%).
- **PassengerId**, **Name**, **Ticket** – удалим как неинформативные для моделирования.

In [18]:
// Удаляем столбцы, которые не будем использовать
df.Columns.Remove("PassengerId");
df.Columns.Remove("Name");
df.Columns.Remove("Ticket");
df.Columns.Remove("Cabin");

Console.WriteLine("Оставшиеся столбцы: " + string.Join(", ", df.Columns.Select(c => c.Name)));

Оставшиеся столбцы: Survived, Pclass, Sex, Age, SibSp, Parch, Fare, Embarked


In [19]:
var ageCol = df["Age"];

// Собираем только те значения, которые не null
var validAges = new List<float>();
for (long i = 0; i < df.Rows.Count; i++)
{
    var obj = ageCol[i];
    if (obj is not null)  // или obj is float f
    {
        validAges.Add((float)obj);
    }
}

// Вычисляем медиану
validAges.Sort();
float medianAge = validAges[validAges.Count / 2];
Console.WriteLine($"Медиана Age: {medianAge}");

// Заменяем null на медиану
for (long i = 0; i < df.Rows.Count; i++)
{
    if (ageCol[i] is null)
        ageCol[i] = medianAge;
}

// Проверка
long ageNulls = 0;
for (long i = 0; i < df.Rows.Count; i++)
    if (ageCol[i] is null)
        ageNulls++;
Console.WriteLine($"Пропусков в Age после заполнения: {ageNulls}");

Медиана Age: 28
Пропусков в Age после заполнения: 0


In [20]:
var embarkedCol = df["Embarked"];
var modeEmbarked = embarkedCol.Cast<string>()
    .Where(s => s != null)
    .GroupBy(s => s)
    .OrderByDescending(g => g.Count())
    .First().Key;
Console.WriteLine($"Мода Embarked: {modeEmbarked}");

for (long i = 0; i < df.Rows.Count; i++)
{
    if (embarkedCol[i] is null)
        embarkedCol[i] = modeEmbarked;
}

Console.WriteLine("Пропуски в Embarked заполнены.");

Мода Embarked: S
Пропуски в Embarked заполнены.


## 3. Кодирование категориальных признаков

Выполним:
- **Label Encoding** для бинарного признака `Sex` (female → 0, male → 1).
- **One-Hot Encoding** для `Embarked` (S, C, Q) – создадим три новых столбца с 0/1.
- Столбец `Pclass` можно оставить как числовой (1,2,3) или тоже закодировать. Оставим как есть.

In [21]:
var sexCol = df["Sex"];
var sexValues = new int[df.Rows.Count];
for (long i = 0; i < df.Rows.Count; i++)
{
    sexValues[i] = (string)sexCol[i] == "male" ? 1 : 0;
}

// Удаляем исходный столбец и добавляем новый числовой
df.Columns.Remove("Sex");
var sexNewCol = new PrimitiveDataFrameColumn<int>("Sex", sexValues);
df.Columns.Insert(df.Columns.Count, sexNewCol);

Console.WriteLine("Готово. Уникальные значения Sex: " + string.Join(", ", sexValues.Distinct()));

Готово. Уникальные значения Sex: 1, 0


In [22]:
var embarkedOrig = df["Embarked"].Cast<string>().ToArray();
var categories = new[] { "S", "C", "Q" };

// Создаём отдельные столбцы для каждой категории
foreach (var cat in categories)
{
    var values = embarkedOrig.Select(v => v == cat ? 1 : 0).ToArray();
    var colName = "Embarked_" + cat;
    var newCol = new PrimitiveDataFrameColumn<int>(colName, values);
    df.Columns.Add(newCol);
}

// Удаляем исходный столбец Embarked
df.Columns.Remove("Embarked");

Console.WriteLine("Добавлены столбцы: " + string.Join(", ", categories.Select(c => "Embarked_" + c)));
df.Head(5).Display();

Добавлены столбцы: Embarked_S, Embarked_C, Embarked_Q


index,Survived,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked_S,Embarked_C,Embarked_Q
0,0,3,22,1,0,7.25,1,1,0,0
1,1,1,38,1,0,71.2833,0,0,1,0
2,1,3,26,0,0,7.925,0,1,0,0
3,1,1,35,1,0,53.1,0,1,0,0
4,0,3,35,0,0,8.05,1,1,0,0


## 4. Масштабирование числовых признаков

Применим два подхода на примере столбца `Fare`:
1. **StandardScaler** (Z‑score): вычитаем среднее и делим на стандартное отклонение.
2. **MinMaxScaler**: приводим значения в диапазон [0, 1].

Также промасштабируем `Age` с помощью StandardScaler.

Для сравнения построим гистограммы до и после.

In [23]:
var fareCol = df["Fare"];
var fareValues = new float[df.Rows.Count];
for (long i = 0; i < df.Rows.Count; i++)
{
    object val = fareCol[i];
    if (val is float f)
        fareValues[i] = f;
    else if (val is string s)
        fareValues[i] = float.Parse(s, CultureInfo.InvariantCulture);
    else
        fareValues[i] = float.NaN; // на случай, если встретится другой тип
}

double meanFare = fareValues.Average();
double stdFare = Math.Sqrt(fareValues.Average(v => Math.Pow(v - meanFare, 2)));
double minFare = fareValues.Min();
double maxFare = fareValues.Max();

Console.WriteLine($"Fare: mean={meanFare:F2}, std={stdFare:F2}, min={minFare:F2}, max={maxFare:F2}");

Fare: mean=32,20, std=49,67, min=0,00, max=512,33


In [24]:
// Функция для стандартизации столбца
void StandardizeColumnUnsafe(DataFrame df, string colName, double mean, double std)
{
    var col = df[colName];
    for (long i = 0; i < df.Rows.Count; i++)
    {
        float x;
        object val = col[i];
        if (val is float f)
            x = f;
        else if (val is string s)
            x = float.Parse(s, CultureInfo.InvariantCulture);
        else
            continue; // пропускаем нераспознанные значения
        
        col[i] = (float)((x - mean) / std);
    }
}

// Стандартизируем Fare
StandardizeColumn(df, "Fare", meanFare, stdFare);

// Аналогично для Age: вычислим статистики и стандартизируем
var ageVals = df["Age"].Cast<float>().ToArray();
double meanAge = ageVals.Average();
double stdAge = Math.Sqrt(ageVals.Average(a => Math.Pow(a - meanAge, 2)));
StandardizeColumn(df, "Age", meanAge, stdAge);

Console.WriteLine("Столбцы Fare и Age стандартизированы (StandardScaler).");
Console.WriteLine($"Новые Fare: среднее ≈ {df["Fare"].Cast<float>().Average():F4}, стд ≈ {Math.Sqrt(df["Fare"].Cast<float>().ToArray().Average(x => Math.Pow(x - df["Fare"].Cast<float>().Average(), 2))):F4}");

Столбцы Fare и Age стандартизированы (StandardScaler).
Новые Fare: среднее ≈ -0,0000, стд ≈ 1,0000


In [25]:
// Построим гистограммы до и после (данные до масштабирования сохранены в fareValues)
var histBefore = PlotlyCS.Histogram<float, float, string>(X: fareValues)
    .WithTitle("Fare до масштабирования (исходный)")
    .WithXAxisStyle(Title.init("Fare"));

var histAfter = PlotlyCS.Histogram<float, float, string>(X: df["Fare"].Cast<float>().ToArray())
    .WithTitle("Fare после StandardScaler")
    .WithXAxisStyle(Title.init("Fare (Z-score)"));

display(histBefore);
display(histAfter);

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

## 5. Заключение

В ходе лабораторной работы:
- Загружен датасет Titanic, содержащий пропуски и категориальные признаки.
- Проведён анализ пропусков: столбцы с большим количеством пропусков удалены, числовой признак Age заполнен медианой, категориальный Embarked – модой.
- Выполнено кодирование: бинарный Sex преобразован в метки 0/1, для Embarked применён One‑Hot Encoding.
- Числовые признаки Age и Fare масштабированы методом Z‑score (StandardScaler). Визуализация гистограмм подтверждает изменение распределения (сдвиг и масштабирование).

Полученный обработанный датафрейм готов к следующему этапу – построению моделей машинного обучения.

In [26]:
Console.WriteLine("Итоговый DataFrame:");
df.Head(10).Display();

Итоговый DataFrame:


index,Survived,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked_S,Embarked_C,Embarked_Q
0,0,3,-0.5657364,1,0,-0.50244516,1,1,0,0
1,1,1,0.6638611,1,0,0.7868453,0,0,1,0
2,1,3,-0.25833702,0,0,-0.48885426,0,1,0,0
3,1,1,0.43331155,1,0,0.4207302,0,1,0,0
4,0,3,0.43331155,0,0,-0.48633742,1,1,0,0
5,0,3,-0.10463735,0,0,-0.47811645,1,0,0,1
6,0,1,1.8934586,0,0,0.39581352,1,1,0,0
7,0,3,-2.1027334,3,1,-0.22408311,1,1,0,0
8,1,3,-0.18148719,0,2,-0.42425615,0,1,0,0
9,1,2,-1.1805352,1,0,-0.042955495,0,0,1,0
